# M02-01 — Ingesta CSV y JSON

[← Anterior](01-teoria.ipynb) · [Siguiente →](03-lab-schema-tipos.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Cargar customers, orders, products y events y comprobar los volúmenes canónicos. Casi sin transformar.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M02-01-ingesta-csv-json.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Arranque y sesión

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Celda 0 + sesión. Compruebo que RAW existe antes de leer.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `True` y una sesión `local[*]`.

**Por qué este paso.** Todas las lecturas de este curso son rutas locales del repo (`RAW`, no un string suelto).


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m02")
print(RAW.exists())


### Paso 2 — CSV de clientes y pedidos

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Leo CSV con header. Sin schema: todo string. Cuento y miro 3 filas de orders.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `customers 250` · `orders 800`. Schema de `orders` con `OrderId`, `CustomerId`, `OrderDate`, `Status`, `Channel` (todo `string`).

**Por qué este paso.** Sin schema ves el fichero crudo. Si cuentas 801, has contado la cabecera.

**Si no sale.** `option("header", True)` en los dos.


In [ ]:
customers = spark.read.option("header", True).csv(str(RAW / "customers.csv"))
orders = spark.read.option("header", True).csv(str(RAW / "orders.csv"))
print("customers", customers.count(), "orders", orders.count())
orders.printSchema()
orders.show(3, truncate=False)


### Paso 3 — JSON array y JSONL

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

products.json es un array: multiLine=True. events.jsonl es una línea = un objeto.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `products 60` · `events 2500`. En productos aparecen `productId` y `listPrice` (camelCase).

**Por qué este paso.** El API es el mismo (`.json`); cambia el fichero.

**Si no sale.** Si products = 362 o ves `_corrupt_record`: falta `multiLine=True`.


In [ ]:
products = spark.read.option("multiLine", True).json(str(RAW / "products.json"))
events = spark.read.json(str(RAW / "events.jsonl"))
print("products", products.count(), "events", events.count())
products.printSchema()
events.printSchema()


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Cuentas las cuatro fuentes otra vez (Run All). Debes tener **250 / 800 / 60 / 2500**.
Cada lectura tiene su celda Markdown encima.


## Mejora — Líneas de pedido

Lee `order_items.csv` con header, cuenta y muestra 3 filas. Markdown + código + ejecuta.

Si te atasca, el código está en la celda siguiente.


In [ ]:
items = spark.read.option("header", True).csv(str(RAW / "order_items.csv"))
print(items.count())  # 2046
items.show(3)


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| PATH not found | Saltaste la Celda 0 | Pega el arranque y usa `RAW` |
| orders = 801 | Contaste la cabecera | `option("header", True)` |
| products 362 / `_corrupt_record` | JSON array sin multiLine | `option("multiLine", True)` |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M02-02 schema y tipos](03-lab-schema-tipos.ipynb).
